# Follicle counting

This notebook uses a helper script with DBSCAN to:

* Count putative individual B cell follicles in selected, annotated BANKSY domains
* Plot results in barplots and boxplots
* Save a summary of follicle counts across samples

Imports and setup:

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from pathlib import Path
import os
import hdbscan

import importlib
import scripts.follicle_counting_helpers as fh
importlib.reload(fh)

### Define base names and find files

In [ ]:
DATA_DIR = Path("/Users/emmyberg/Documents/IHOPE_SpatialProteomics/data/anndata/zscore_log2/celltyped/follicledomains")
OUT_DIR = Path("../results/reports/follicle_counts")
OUT_DIR.mkdir(exist_ok=True, parents=True)

suffix = "_celltypes_follicledomains.h5ad"
basenames = sorted(
    f.name[: -len(suffix)]
    for f in DATA_DIR.glob(f"*{suffix}")
)

### Apply loop to all samples of interest

In [ ]:
results = []

for basename in basenames:
    try:
        print(f"\nProcessing {basename}")

        filepath = (
            DATA_DIR /
            f"{basename}_celltypes_follicledomains.h5ad"
        )

        adata = sc.read_h5ad(filepath)

        mask = (
            (adata.obs["type_B"] == True) &
            (adata.obs["B_follicle"] == True)
        )

        print(f"{basename}: {mask.sum()} B cells in follicle regions")

        adata, n_follicles = fh.detect_follicles(adata)

        fh.plot_follicles(adata, basename, OUT_DIR)

        print(f"Plotted {basename}, saved as {OUT_DIR / f'{basename}_follicles.png'}")

        adata.write(
            OUT_DIR / f"{basename}_follicle_clustered.h5ad"
        )

        immune_mask = (
            adata.obs["type_B"]
            | adata.obs["type_T"]
            | adata.obs["type_NK"]
            | adata.obs["type_Myeloid"]
        )

        results.append({
            "sample": basename,
            "n_follicles": n_follicles,
            "n_candidate_Bcells": mask.sum(),
            "total_cells": adata.n_obs,
            "total_immune_cells": int(immune_mask.sum())
        })

    except Exception as e:
        print(f"Failed on {basename}: {e}")

In [ ]:
results_df = pd.DataFrame(results)

results_df = fh.normalize_follicle_counts(results_df, count_col="total_immune_cells")

results_df.to_csv(
    OUT_DIR / "follicle_counts_summary.csv",
    index=False
)

results_df

**Follicle counts per sample (ranked) barplot**

### Filtering out bad clusters

...and logging the process

In [ ]:
import importlib
import scripts.follicle_counting_helpers as fh
importlib.reload(fh)

In [ ]:
diagnostic_dir = OUT_DIR / "diagnostic_numbered"
diagnostic_dir.mkdir(exist_ok=True, parents=True)

for basename in basenames:
    adata = sc.read_h5ad(OUT_DIR / f"{basename}_follicle_clustered.h5ad")
    fh.plot_follicles_numbered(adata, basename, diagnostic_dir)

In [ ]:
excluded_clusters = {
    "IHOPE14_MedLN_BottomLeft": [],
    "IHOPE14_MedLN_BottomRight": [],
    "IHOPE14_MedLN_TopRight": [],
    "IHOPE14_MesLN": [],
    "IHOPE20_MedLN": [],
    "IHOPE20_Spleen": [],
    "IHOPE26_MedLN": [],
    "IHOPE26_Spleen": [],
    "IHOPE27_MedLN": [],
    "IHOPE27_Spleen": [],
    "IHOPE39_MedLN": [],
    "IHOPE39_MesLN_A": [],
    "IHOPE39_MesLN_B": [],
    "IHOPE39_Spleen": [54, 68, 65],
}

In [ ]:
exclusion_log = []

for sample, labels_to_drop in excluded_clusters.items():
    path = OUT_DIR / f"{sample}_follicle_clustered.h5ad"
    adata = sc.read_h5ad(path)

    adata, n_removed, n_follicles = fh.apply_cluster_exclusions(adata, labels_to_drop)

    for label in labels_to_drop:
        exclusion_log.append({
            "sample": sample,
            "excluded_label": label,
        })

    adata.write(path)
    fh.plot_follicles(adata, sample, OUT_DIR)

    results_df.loc[results_df["sample"] == sample, "n_follicles"] = n_follicles

exclusion_log_df = pd.DataFrame(exclusion_log)
exclusion_log_df.to_csv(OUT_DIR / "follicle_exclusions_log.csv", index=False)

results_df = fh.normalize_follicle_counts(results_df, count_col="total_immune_cells")
results_df.to_csv(OUT_DIR / "follicle_counts_summary.csv", index=False)

results_df

### Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

df_sorted = results_df.sort_values("n_follicles", ascending=False)

sns.barplot(
    data=df_sorted,
    x="sample",
    y="n_follicles",
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of follicles")
plt.xlabel("Sample")
plt.title("Follicle counts per sample (ranked)")
plt.tight_layout()
plt.grid(False)


plt.savefig(
    OUT_DIR / "follicle_counts_ranked_barplot.png",
    dpi=300
)

plt.show()

**Normalized follicle count**



In [ ]:
plt.figure(figsize=(12, 6))

df_sorted_norm = results_df.sort_values("follicles_per_10000_immune_cells", ascending=False)

sns.barplot(
    data=df_sorted_norm,
    x="sample",
    y="follicles_per_10000_immune_cells",
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Follicles per 10,000 immune cells")
plt.xlabel("Sample")
plt.title("Normalized follicle counts per sample (ranked)")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_ranked_barplot_normalized.png",
    dpi=300
)

plt.show()

**Follicle counts per sample (custom order) barplot**


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Define order manually
custom_order = [
    "IHOPE14_MedLN_BottomLeft",
    "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MedLN_TopRight",
    "IHOPE20_MedLN",
    "IHOPE26_MedLN",
    "IHOPE27_MedLN",
    "IHOPE39_MedLN",
    "IHOPE14_MesLN",
    "IHOPE39_MesLN_A",
    "IHOPE39_MesLN_B",
    "IHOPE20_Spleen",
    "IHOPE26_Spleen",
    "IHOPE27_Spleen",
    "IHOPE39_Spleen"
]

sns.barplot(
    data=results_df,
    x="sample",
    y="n_follicles",
    order=custom_order,
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Number of follicles")
plt.xlabel("Sample")
plt.title("Follicle counts per sample")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_alphabetical_order_barplot.png",
    dpi=300
)

plt.show()

**Normalized**

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_df,
    x="sample",
    y="follicles_per_10000_immune_cells",
    order=custom_order,
    color="steelblue"
)

plt.xticks(rotation=60, ha="right")
plt.ylabel("Follicles per 10,000 immune cells")
plt.xlabel("Sample")
plt.title("Normalized follicle counts per sample")
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_alphabetical_order_barplot_normalized.png",
    dpi=300
)

plt.show()

**Follicle counts per sample (grouped by tissue manually) boxplot**


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

tissue_map = {
    # MedLN
    "IHOPE14_MedLN_BottomLeft": "MedLN",
    "IHOPE14_MedLN_BottomRight": "MedLN",
    "IHOPE14_MedLN_TopRight": "MedLN",
    "IHOPE20_MedLN": "MedLN",
    "IHOPE26_MedLN": "MedLN",
    "IHOPE27_MedLN": "MedLN",
    "IHOPE39_MedLN": "MedLN",

    # MesLN
    "IHOPE14_MesLN": "MesLN",
    "IHOPE39_MesLN_A": "MesLN",
    "IHOPE39_MesLN_B": "MesLN",

    # Spleen
    "IHOPE20_Spleen": "Spleen",
    "IHOPE26_Spleen": "Spleen",
    "IHOPE27_Spleen": "Spleen",
    "IHOPE39_Spleen": "Spleen"
}

results_df["tissue"] = results_df["sample"].map(tissue_map)

In [ ]:
donor_map = {
    "IHOPE14_MedLN_BottomLeft": "IHOPE14",
    "IHOPE14_MedLN_BottomRight": "IHOPE14",
    "IHOPE14_MedLN_TopRight": "IHOPE14",
    "IHOPE20_MedLN": "IHOPE20",
    "IHOPE26_MedLN": "IHOPE26",
    "IHOPE27_MedLN": "IHOPE27",
    "IHOPE39_MedLN": "IHOPE39",

    "IHOPE14_MesLN": "IHOPE14",
    "IHOPE39_MesLN_A": "IHOPE39",
    "IHOPE39_MesLN_B": "IHOPE39",

    "IHOPE20_Spleen": "IHOPE20",
    "IHOPE26_Spleen": "IHOPE26",
    "IHOPE27_Spleen": "IHOPE27",
    "IHOPE39_Spleen": "IHOPE39",
}

results_df["donor"] = results_df["sample"].map(donor_map)

donor_ids = sorted(results_df["donor"].unique())
donor_colors = dict(zip(donor_ids, sns.color_palette("colorblind", n_colors=len(donor_ids))))

In [ ]:
plt.figure(figsize=(8,6))

tissue_order = ["MedLN", "MesLN", "Spleen"]

sns.boxplot(
    data=results_df,
    x="tissue",
    y="n_follicles",
    order=tissue_order,
    color="0.85",
    showfliers=False
)

sns.stripplot(
    data=results_df,
    x="tissue",
    y="n_follicles",
    order=tissue_order,
    hue="donor",
    palette=donor_colors,
    legend=False,
    jitter=0.15,
    s=6
)

handles = [
    plt.Line2D([0], [0], marker="o", color="white", markerfacecolor=color,
               markersize=8, label=donor)
    for donor, color in donor_colors.items()
]
plt.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
           frameon=False, title="Donor")

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.ylabel("Number of follicles")
plt.title("Follicle counts by tissue")
plt.xlabel(None)
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_boxplot_tissue.png",
    dpi=300
)

plt.show()

**Normalized**

In [ ]:
plt.figure(figsize=(8,6))

sns.boxplot(
    data=results_df,
    x="tissue",
    y="follicles_per_10000_immune_cells",
    order=tissue_order,
    color="0.85",
    showfliers=False
)

sns.stripplot(
    data=results_df,
    x="tissue",
    y="follicles_per_10000_immune_cells",
    order=tissue_order,
    hue="donor",
    palette=donor_colors,
    legend=False,
    jitter=0.15,
    s=6
)

handles = [
    plt.Line2D([0], [0], marker="o", color="white", markerfacecolor=color,
               markersize=8, label=donor)
    for donor, color in donor_colors.items()
]
plt.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
           frameon=False, title="Donor")

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.ylabel("Follicles per 10,000 immune cells")
plt.title("Normalized follicle counts by tissue")
plt.xlabel(None)
plt.tight_layout()
plt.grid(False)

plt.savefig(
    OUT_DIR / "follicle_counts_boxplot_tissue_normalized.png",
    dpi=300
)

plt.show()

# Summary

Generate a full summary based on the cell type summary ("type" level) and the follicle counts.

In [ ]:
follicle_df = pd.read_csv(
    OUT_DIR / "follicle_counts_summary.csv"
)

follicle_df.head()

In [ ]:
summary_dir = Path("../results/reports/zscore_log2")
summary_files = list(summary_dir.glob("celltype_summary_*.csv"))
summary_files

In [ ]:
all_rows = []

for file in summary_files:

    df = pd.read_csv(file)

    sample_name = file.stem.replace("celltype_summary_", "")

    type_df = df[df["level"] == "type"]

    row = {"sample": sample_name}

    if len(type_df) > 0:
        row["total_cells"] = type_df["total_cells"].iloc[0]

    for _, r in type_df.iterrows():
        ct = r["cell_type"]

        row[f"{ct}_count"] = r["n_cells"]
        row[f"{ct}_pct"] = r["pct_total"]

    all_rows.append(row)

celltype_wide = pd.DataFrame(all_rows)

celltype_wide.head()

In [ ]:
final_report = follicle_df.merge(
    celltype_wide,
    on="sample",
    how="left"
)

final_report

In [ ]:
final_report.to_csv(
    OUT_DIR / "follicle_celltype_report.csv",
    index=False
)

final_report